In [3]:
import polars as pl
import os
import glob
from datetime import datetime

# =====================================================================
# GLOBAL PATH CONFIGURATION (DYNAMIC PATH)
# =====================================================================
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"

if not os.path.exists(test_path):
    raise FileNotFoundError(f"Root directory not found: {test_path}")

base_path = f"{test_path}/WFM-Expedia-HCM - Branding files"

folder_paths = {
    "roster_path": f"{base_path}/Schedule/Schedule (Ops version)/2026/Master_Schedule_Merged.xlsx",
    "ot_plan_path": f"{base_path}/OT/VN Expedia - OT Plan.xlsx",
    "ramco_ot_csv_path": f"{base_path}/BI_Task/CODE/Python_Code/ramco_ot.csv",
    "rta_report_folder": f"{base_path}/Rawdata/STORAGE_OUTPUT_RTA",
    "rta_intervals_folder": f"{base_path}/Rawdata/OUTPUT_AGENT_ACTIVITY_INTERVALS",
    "output_ot_path": f"{base_path}/Rawdata/OUTPUT_OT_REGISTRATION_REPORT.xlsx"
}

roster_path = folder_paths["roster_path"]
ot_plan_path = folder_paths["ot_plan_path"]
ramco_ot_csv_path = folder_paths["ramco_ot_csv_path"]
rta_report_folder = folder_paths["rta_report_folder"]
rta_intervals_folder = folder_paths["rta_intervals_folder"]
output_ot_path = folder_paths["output_ot_path"]

# HELPER FUNCTION: Simulate MROUND function in Excel/DAX
def mround(col_expr, multiple):
    return ((col_expr / multiple) + 1e-9).round(0) * multiple

print("STARTING OT REGISTRATION REPORT PROCESSING...")
print("-" * 60)

# =====================================================================
# [1/10] READ ROSTER DATA AND UNPIVOT
# =====================================================================
print("[1/10] Processing Roster schedule data...")
roster_df = pl.read_excel(roster_path, sheet_name="Sheet1").rename({"OracleID": "Emp ID", "Employee Name": "Agent Name"})

id_vars = [c for c in ["IEX ID", "IEX", "Emp ID", "Email", "Agent Name", "Status", "Team"] if c in roster_df.columns]
date_cols = [c for c in roster_df.columns if c not in id_vars]

roster_melted = roster_df.unpivot(index=id_vars, on=date_cols, variable_name="Scheduled_Date_Str", value_name="Shift")
roster_melted = roster_melted.with_columns([
    pl.col("Scheduled_Date_Str").str.slice(0, 10).str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("Scheduled_Date"),
    pl.when(pl.col("Shift") == "-").then(pl.lit("OFF")).otherwise(pl.col("Shift")).fill_null("OFF").alias("Shift")
])

# =====================================================================
# [2/10] READ OT REGISTRATION DATA (OT PLAN)
# =====================================================================
print("[2/10] Processing OT Plan registration data...")
ot_df = pl.read_excel(ot_plan_path, sheet_name="OT Register").filter(pl.col("Emp ID").is_not_null())
ot_df = ot_df.with_columns([
    pl.col("OT Date").cast(pl.Date, strict=False),
    pl.col("Start Date Time").cast(pl.Datetime, strict=False),
    pl.col("End Date Time").cast(pl.Datetime, strict=False),
    pl.col("Duration").cast(pl.Float64, strict=False)
])

# =====================================================================
# [3/10] JOIN TO GET SHIFT DATA (CURRENT & PREVIOUS DAY)
# =====================================================================
print("[3/10] Joining current and previous day shift data...")
roster_key = roster_melted.select(["Emp ID", "Email", "Scheduled_Date", "Shift"])

# Shift on OT registration day
ot_processed = ot_df.join(
    roster_key, left_on=["Emp ID", "OT Date"], right_on=["Emp ID", "Scheduled_Date"], how="left"
).with_columns(pl.col("Shift").fill_null("OFF"))

# Previous day shift (minus 1 day)
ot_processed = ot_processed.with_columns((pl.col("OT Date") - pl.duration(days=1)).alias("Prev OT Date"))
ot_processed = ot_processed.join(
    roster_key.select(["Emp ID", "Scheduled_Date", "Shift"]).rename({"Shift": "Prev_Day_Shift"}),
    left_on=["Emp ID", "Prev OT Date"], right_on=["Emp ID", "Scheduled_Date"], how="left"
).with_columns(pl.col("Prev_Day_Shift").fill_null("OFF"))

# =====================================================================
# [4/10] DETERMINE SHIFT DATE & CLASSIFY OT
# =====================================================================
print("[4/10] Allocating night shifts and classifying OT (PO/Regular)...")
def check_night_shift(shift_col):
    split_shift = shift_col.str.split_exact("-", 1)
    has_dash = shift_col.str.contains("-").fill_null(False)
    start_num = split_shift.struct.field("field_0").str.strip_chars().cast(pl.Int32, strict=False).fill_null(0)
    end_num = split_shift.struct.field("field_1").str.strip_chars().cast(pl.Int32, strict=False).fill_null(2400)
    return pl.when(has_dash).then(end_num <= start_num).otherwise(pl.lit(False))

ot_processed = ot_processed.with_columns([
    pl.col("Start Date Time").dt.date().alias("otStartDate"),
    pl.col("Start Date Time").dt.hour().alias("otStartHour")
])

# Logic to shift date for night shifts
ot_processed = ot_processed.with_columns(
    pl.when(pl.col("otStartDate") > pl.col("OT Date")).then(pl.col("OT Date"))
    .when(pl.col("otStartDate") == pl.col("OT Date"))
    .then(
        pl.when(check_night_shift(pl.col("Prev_Day_Shift")) & (pl.col("otStartHour") < 14))
        .then(pl.col("OT Date") - pl.duration(days=1))
        .otherwise(pl.col("OT Date"))
    )
    .otherwise(pl.col("otStartDate")).alias("Shift Date")
)

ot_processed = ot_processed.with_columns(
    pl.when(pl.col("Shift Date") == pl.col("OT Date")).then(pl.col("Shift")).otherwise(pl.col("Prev_Day_Shift")).alias("Base Shift")
).with_columns(
    pl.when(pl.col("Base Shift").str.contains("-")).then(pl.lit("Regular OT")).otherwise(pl.lit("PO")).alias("OT Type")
)

# =====================================================================
# [5/10] CREATE FINAL SHIFT STRING FOR REPORT DISPLAY
# =====================================================================
print("[5/10] Creating Final Shift string...")
po_agg = ot_processed.filter(pl.col("OT Type") == "PO").group_by(["Emp ID", "Shift Date"]).agg([
    pl.col("Start Date Time").min().alias("PO_Min_Start"),
    pl.col("Duration").sum().alias("PO_Total_Hours")
])
ot_processed = ot_processed.join(po_agg, on=["Emp ID", "Shift Date"], how="left")

split_base = pl.col("Base Shift").str.split_exact("-", 1)
has_dash = pl.col("Base Shift").str.contains("-").fill_null(False)

base_s_str = pl.when(has_dash).then(split_base.struct.field("field_0").str.strip_chars().str.zfill(4)).otherwise(pl.lit("0000"))
base_e_str = pl.when(has_dash).then(split_base.struct.field("field_1").str.strip_chars().str.zfill(4)).otherwise(pl.lit("0000"))
base_s_dt = pl.col("Shift Date").dt.combine(base_s_str.str.strptime(pl.Time, "%H%M", strict=False))
base_e_time = base_e_str.str.strptime(pl.Time, "%H%M", strict=False)
base_e_dt = pl.when(base_e_time < base_s_str.str.strptime(pl.Time, "%H%M", strict=False)).then((pl.col("Shift Date") + pl.duration(days=1)).dt.combine(base_e_time)).otherwise(pl.col("Shift Date").dt.combine(base_e_time))

reg_final_start = pl.min_horizontal("Start Date Time", base_s_dt)
reg_final_end = pl.max_horizontal("End Date Time", base_e_dt)
reg_final_shift = reg_final_start.dt.strftime("%H%M") + "-" + reg_final_end.dt.strftime("%H%M")

po_final_end_dt = pl.col("PO_Min_Start") + pl.duration(minutes=(pl.col("PO_Total_Hours").fill_null(0) * 60).cast(pl.Int64))
po_final_shift = pl.col("PO_Min_Start").dt.strftime("%H%M") + "-" + po_final_end_dt.dt.strftime("%H%M")

ot_processed = ot_processed.with_columns(
    pl.when(pl.col("OT Type") == "PO").then(po_final_shift).otherwise(reg_final_shift).alias("Final Shift")
)

# =====================================================================
# [6/10] PRELIMINARY DATA GROUPING BY EMPLOYEE AND DATE
# =====================================================================
print("[6/10] Grouping and aggregating data...")
group_cols = [c for c in ["Week", "Agent/Non-Agent", "TL", "Emp ID", "Email", "Name", "Shift Date", "Base Shift", "Final Shift", "OT Type"] if c in ot_processed.columns]
ot_final = ot_processed.group_by(group_cols).agg([
    pl.col("OT Date").first(),
    pl.col("Start Date Time").min().alias("OT Time Start"),
    pl.col("End Date Time").max().alias("OT Time End")
])

# =====================================================================
# [7/10] EXTRACT AND MATCH WITH RAMCO OT
# =====================================================================
print("[7/10] Reading and matching RAMCO data...")
if os.path.exists(ramco_ot_csv_path):
    RAMCO_OT = pl.read_csv(ramco_ot_csv_path, encoding="utf-8")
    ramco_ot_filtered = RAMCO_OT.filter(pl.col("ot_type").str.contains("OT")).group_by(["EID", "Date"]).agg([
        pl.col("hours").sum().alias("RAMCO_Logged_Hours"),
        pl.col("ot_status").first().alias("RAMCO_Status")
    ]).with_columns([
        pl.col("EID").cast(pl.Int64, strict=False),
        pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d", strict=False)
    ])

    ot_final = ot_final.join(
        ramco_ot_filtered, left_on=["Emp ID", "Shift Date"], right_on=["EID", "Date"], how="left"
    ).with_columns([
        pl.col("RAMCO_Logged_Hours").fill_null(0),
        pl.col("RAMCO_Status").str.extract(r"-\s*([A-Za-z]+)", 1).alias("RAMCO_Approval_Status")
    ])

# =====================================================================
# [8/10] CALCULATE ACTUAL PRODUCTIVE HOURS (RTA & INTERVALS)
# =====================================================================
print("[8/10] Generating 30-min Grid Slots to match Productive hours (Power Query logic)...")

# 8.1 Read and concatenate RTA Report
rta_files = glob.glob(f"{rta_report_folder}/*.xlsx") + glob.glob(f"{rta_report_folder}/*.csv")
if rta_files:
    df_rta_list = [
        pl.read_excel(f, infer_schema_length=0) if f.endswith('.xlsx') else pl.read_csv(f, infer_schema_length=0) 
        for f in rta_files
    ]
    RTA_REPORT = pl.concat(df_rta_list, how='diagonal').with_columns([
        pl.col("OracleID").cast(pl.Int64, strict=False),
        pl.col("Date_Converted").str.slice(0, 10).str.strptime(pl.Date, "%Y-%m-%d", strict=False),
        pl.col("start").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
        pl.col("end").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
        pl.col("sum_productive").cast(pl.Float64, strict=False),
        pl.col("OT PreShift").cast(pl.Float64, strict=False),
        pl.col("OT PostShift").cast(pl.Float64, strict=False)
    ])
    ot_final = ot_final.join(
        RTA_REPORT.select(['Date_Converted', 'OracleID', 'OT PreShift', 'OT PostShift', 'start', 'end', 'sum_productive']), 
        left_on=["Shift Date", "Emp ID"], right_on=["Date_Converted", "OracleID"], how="left"
    )

# 8.2 Read and extract Agent Activity Intervals grid
interval_files = glob.glob(f"{rta_intervals_folder}/*.csv")
if interval_files:
    df_intervals = pl.concat([pl.read_csv(f, infer_schema_length=0) for f in interval_files], how='diagonal')
    
    # Filter Productive = Yes and group by Email + Datetime
    prod_int = df_intervals.filter(pl.col("Productive Aux Flag (Yes / No)") == "Yes").with_columns([
        pl.col("VNT_Intervals").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
        pl.col("Duration").cast(pl.Float64, strict=False)
    ]).group_by(["Email Id", "VNT_Intervals"]).agg(pl.col("Duration").sum().alias("Interval_Productive"))

    # Split into 30-minute Grid Slots for the OT registration period
    ot_slots = ot_processed.select(["Emp ID", "Email", "Shift Date", "Start Date Time", "End Date Time"]).drop_nulls()
    ot_slots = ot_slots.with_columns(
        pl.datetime_ranges(start=pl.col("Start Date Time").dt.truncate("30m"), end=pl.col("End Date Time"), interval="30m", closed="left").alias("VNT_Intervals")
    ).explode("VNT_Intervals")
    
    # Determine start and end points of each slot to calculate accurate registered Duration
    ot_slots = ot_slots.with_columns((pl.col("VNT_Intervals") + pl.duration(minutes=30)).alias("Grid_Slot_End")).with_columns([
        pl.max_horizontal("VNT_Intervals", "Start Date Time").alias("S_Start"),
        pl.min_horizontal("Grid_Slot_End", "End Date Time").alias("S_End")
    ]).with_columns(((pl.col("S_End") - pl.col("S_Start")).dt.total_milliseconds() / 3_600_000).alias("S_Dur"))

    # Join with Productive table to get actual numbers and Group by Employee
    joined = ot_slots.join(prod_int, left_on=["Email", "VNT_Intervals"], right_on=["Email Id", "VNT_Intervals"], how="left")
    ot_agg = joined.group_by(["Emp ID", "Shift Date"]).agg([
        pl.col("S_Dur").sum().alias("OT Registered"),
        pl.col("Interval_Productive").sum().alias("OT Actual Productive")
    ])
    
    # Append results to final table and subtract to get Base Shift Actual
    ot_final = ot_final.join(ot_agg, on=["Emp ID", "Shift Date"], how="left").with_columns([
        pl.col("OT Registered").fill_null(0),
        pl.col("OT Actual Productive").fill_null(0),
        pl.max_horizontal(pl.col("sum_productive").fill_null(0) - pl.col("OT Actual Productive").fill_null(0), 0).alias("Base Shift Actual Productive")
    ])

# =====================================================================
# [9/10] CALCULATE CAPPING & OT CONFIRM (DAX LOGIC)
# =====================================================================
print("[9/10] Applying Capping rules and calculating OT Confirm (DAX Logic)...")

# 1. Capping for RAMCO & BACKEND
ot_final = ot_final.with_columns([
    pl.when(pl.col("OT Type") == "Regular OT").then(pl.min_horizontal(pl.col("OT Registered"), 4))
    .when(pl.col("OT Type") == "PO").then(pl.min_horizontal(pl.col("OT Registered"), 12))
    .otherwise(0).alias("RAMCO OT")
])
ot_final = ot_final.with_columns((pl.col("OT Registered") - pl.col("RAMCO OT")).alias("BACKEND OT"))

# 2. OT Confirm Logic (Simulating DAX function)
ot_final = ot_final.with_columns([
    pl.col("OT Registered").alias("_TotalSlot"),
    pl.col("OT Actual Productive").alias("_TotalProductive")
])

ot_final = ot_final.with_columns([
    mround(pl.col("_TotalProductive"), 0.25).alias("_RoundedProd"),
    ((pl.col("_TotalSlot") / 4).floor() * 0.25).alias("_InitialBreak")
])

ot_final = ot_final.with_columns(pl.max_horizontal(pl.col("_TotalProductive") - pl.col("_InitialBreak"), 0).alias("_EffectiveHours"))

ot_final = ot_final.with_columns(
    pl.when(pl.col("_TotalProductive") < pl.col("_TotalSlot"))
    .then((pl.col("_EffectiveHours") / 4).floor() * 0.25)
    .otherwise(pl.col("_InitialBreak")).alias("_FinalBreak")
)

ot_final = ot_final.with_columns([
    mround(pl.col("_RoundedProd") + pl.col("_FinalBreak"), 0.5).alias("_FinalRounded"),
    (pl.col("_TotalSlot") - pl.when(pl.col("_TotalSlot") >= 9).then(1).otherwise(0)).alias("_MaxPayable")
])

ot_final = ot_final.with_columns(
    pl.when((pl.col("_TotalProductive") == 0) | pl.col("_TotalProductive").is_null()).then(0)
    .otherwise(pl.min_horizontal(pl.col("_FinalRounded"), pl.col("_MaxPayable"))).alias("OT Confirm")
)

# Clean up temporary columns
cols_to_drop = [c for c in ot_final.columns if c.startswith("_")] + ["Email Id", "Agent Email", "Total OT Hours"]
ot_final = ot_final.drop([c for c in cols_to_drop if c in ot_final.columns])

# =====================================================================
# [10/10] EXPORT FINAL REPORT FILE
# =====================================================================
print("[10/10] Exporting report...")
ot_final.write_excel(output_ot_path)

print("-" * 60)
print(f"DONE! Data has been successfully exported to:\n>> {output_ot_path}")
print("-" * 60)

# Display a few summary columns for quick check
preview_cols = ["Emp ID", "Shift Date", "OT Registered", "Base Shift Actual Productive", "OT Actual Productive", "OT Confirm"]
print(ot_final.select([c for c in preview_cols if c in ot_final.columns]).head(5))

Could not determine dtype for column 18, falling back to string
Could not determine dtype for column 19, falling back to string


STARTING OT REGISTRATION REPORT PROCESSING...
------------------------------------------------------------
[1/10] Processing Roster schedule data...
[2/10] Processing OT Plan registration data...
[3/10] Joining current and previous day shift data...
[4/10] Allocating night shifts and classifying OT (PO/Regular)...
[5/10] Creating Final Shift string...
[6/10] Grouping and aggregating data...
[7/10] Reading and matching RAMCO data...
[8/10] Generating 30-min Grid Slots to match Productive hours (Power Query logic)...
[9/10] Applying Capping rules and calculating OT Confirm (DAX Logic)...
[10/10] Exporting report...
------------------------------------------------------------
DONE! Data has been successfully exported to:
>> C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_OT_REGISTRATION_REPORT.xlsx
------------------------------------------------------------
shape: (5, 6)
┌───────────┬────────────┬───────────────┬───────────────────────────────┬──────